In [ ]:
# MONTE CARLO SIMULATION
# P = 2 L / (pi D)
# P = probability
# L = needle length (uniform and normal)
# D = distance between lines

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde, norm


def monte_carlo_buffon(N, L_min, L_max, D, L_distribution='uniform', charts=False):
    """
    1. generate random variables
    2. calculate horizontal projection
    3. define intersections
    4. calculate success
    5. calculate probability
    """
    def theoretical_limits():
        # theoretical limits
        t = np.linspace(0, np.pi, 100)
        if L_distribution == 'normal':
            left_std_min = (L_mean - L_std) / 2 * np.sin(t)
            left_std_max = (L_mean + L_std) / 2 * np.sin(t)
            right_std_min = D - left_std_min
            right_std_max = D - left_std_max
            plt.fill_betweenx(t, left_std_min, left_std_max, color='red', alpha=0.3)
            plt.fill_betweenx(t, right_std_min, right_std_max, color='red', alpha=0.3)

            left_limit_min = (L_mean - 3*L_std) / 2 * np.sin(t)
            left_limit_max = (L_mean + 3*L_std) / 2 * np.sin(t)
        else:
            left_limit_min = L_min / 2 * np.sin(t)
            left_limit_max = L_max / 2 * np.sin(t)
        right_limit_min = D - left_limit_min
        right_limit_max = D - left_limit_max
        plt.fill_betweenx(t, left_limit_min, left_limit_max, color='red', alpha=0.3)
        plt.fill_betweenx(t, right_limit_min, right_limit_max, color='red', alpha=0.3)
        
        left_mean = L_mean / 2 * np.sin(t)
        right_mean = D - left_mean
        plt.plot(left_mean, t, color='red', lw=2)
        plt.plot(right_mean, t, color='red', lw=2)
    
    def plot_phase_space():
        plt.figure(figsize = (10, 6))
        # intersection mask
        plt.scatter(x[success_array], theta[success_array], s=2, color='royalblue', label='Touches the line')
        plt.scatter(x[~success_array], theta[~success_array], s=2, color='gray', label='Does not touch the line')
    
        theoretical_limits()

        # plot details
        plt.xlabel('Position (x)')
        plt.ylabel(r'Angle ($\theta$ in radians)')
        plt.yticks(np.linspace(0, np.pi, 5),['0', r'$\pi$/4', r'$\pi$/2', r'$3\pi$/4', r'$\pi$'])
        plt.title(f'Phase Space (N = {N}, L_mean = {L_mean:.1f}, D = {D})')
        plt.legend(loc='lower center')
        plt.show()

    def plot_heatmap():
        plt.figure(figsize = (10, 6))
        plt.hist2d(x, theta, bins=100, weights=success_array.astype(float), cmap='plasma', density=True)

        plt.colorbar(label='Local Probability')
        
        theoretical_limits()

        # plot details
        plt.xlabel('Position (x)')
        plt.ylabel(r'Angle ($\theta$ in radians)')
        plt.yticks(np.linspace(0, np.pi, 5),['0', r'$\pi$/4', r'$\pi$/2', r'$3\pi$/4', r'$\pi$'])
        plt.title(f'Phase Space Heatmap (N = {N}, L_mean = {L_mean:.1f}, D = {D})')
        plt.legend(['Probability Density of Success'], loc='lower center')
        plt.tight_layout()
        plt.show()

    
    def plot_scatter():
        plt.figure(figsize = (10, 6))
        prob_L = 2 * L * success_array / (np.pi * D)

        if L_distribution == 'normal':
            density_model = gaussian_kde(L)
            freq = density_model(L)
        elif L_distribution == 'uniform':
            counts, bin_edges = np.histogram(L, bins=100)
            bin_indices = np.digitize(L, bin_edges[:-1]) - 1 
            freq = counts[bin_indices]
            
        plt.scatter(L[prob_L != 0], prob_L[prob_L != 0], c=freq[prob_L != 0], cmap='plasma', s=10)
        
        # plot details
        plt.xlabel('Neddle Length (L)')
        plt.ylabel('Probability of Success')
        plt.title(f'Scatter (N = {N}, L_mean = {L_mean:.1f}, D = {D})')
        plt.show()

    def plot_success():
        plt.figure(figsize = (10, 6))
        L_clean = L[success_array != 0]
        plt.hist(L, density=True, bins=100, color='gray', alpha=0.5, label='All Needles')
        plt.hist(L_clean, density=True, bins=100, color='royalblue', alpha=0.7, label='Successful Intersections')

        # plot details
        plt.title(f'Comparative Distribution of Successes (N = {N}, L_mean = {L_mean:.1f}, D = {D})')
        plt.xlabel('Neddle Length (L)')
        plt.ylabel('Probability Density')
        plt.grid(axis='y', linestyle='--', alpha=0.7)
        plt.legend()
        plt.show()

    def plot_pi_convergence():
        plt.figure(figsize = (10, 6))
        n = np.linspace(1, N, N)
        P_conv = np.cumsum(success_array)
        L_conv = np.cumsum(L)
        pi_conv = 2 * L_conv[P_conv != 0] / (D * P_conv[P_conv != 0])

        plt.scatter(n[P_conv != 0], pi_conv, color='black', s=10)
        plt.axhline(y=np.pi, color='red', linestyle='--', linewidth=2, label=r'$\pi$ Value')
        plt.title(fr'Estimation of $\pi$ Convergence Curve (N = {N}, L_mean = {L_mean:.1f}, D = {D})')
        plt.xlabel('N Steps')
        plt.ylabel(r'Estimation of $\pi$')
        plt.legend()
        plt.show()
        
    
    # random variables
    x = np.random.uniform(0, D, N)
    theta = np.random.uniform(0, np.pi, N)
    if L_distribution == 'uniform':
        L = np.random.uniform(L_min, L_max, N)
    elif L_distribution == 'normal':
        mean = (L_max + L_min) / 2
        std = (L_max - L_min) / 6
        L = np.random.normal(mean, std, N)
    L_mean = np.mean(L)
    L_std = np.std(L)

    # horizontal projecton
    proj = L/2 * np.sin(theta)

    # intersections
    left_line = x <= proj
    right_line = x >= D - proj

    # success -> left_line U right_line
    success_array = left_line | right_line
    success = np.sum(success_array)

    # probability
    P_simulation = success / N
    P_theoretical = 2 * L_mean / (np.pi * D)

    # estimate Pi
    pi_simulation = 2 * L_mean / (D * P_simulation)

    # prints
    print('-'*40)
    print(f"Sample                 | N = {N:.1f}")
    print(f"Needle lenght (distr)  | {L_distribution.upper()}")
    print(f"Needle lenght (min)    | L = {L_min:.1f}")
    print(f"Needle lenght (max)    | L = {L_max:.1f}")
    print(f"Needle lenght (mean)   | L = {L_mean:.1f}")
    print(f"Needle lenght (std)    | L = {L_std:.1f}")
    print(f"Distance between lines | D = {D:.1f}")
    print(f"========== Probabilities ===============")
    print(f"Simulation             | P = {P_simulation:.6f}")
    print(f"Theoretical            | P = {P_theoretical:.6f}")
    print(f"========= Estimate of Pi ===============")
    print(f"Simulation             | Pi = {pi_simulation:.6f}")
    print(f"Real                   | Pi = {np.pi:.6f}")
    print('-'*40)

    if charts:
        plot_phase_space()
        plot_heatmap()
        plot_scatter()
        plot_success()
        plot_pi_convergence()
    
    # return P_simulation, P_theoretical

In [ ]:
for samples in [10, 100, 1000, 10000, 100000]:
    monte_carlo_buffon(samples, 2, 3, 5)

In [ ]:
monte_carlo_buffon(10000, 2, 3, 5, L_distribution='uniform', charts=True)

In [ ]:
monte_carlo_buffon(10000, 2, 3, 5, L_distribution='normal', charts=True)